# evaluate_checkpoints

## What this notebook does

- **Starts from:** a run that already finished, at the saved checkpoints you name in `STEPS` (fetched from the Hub if
  they are not on this machine). Step `0` is the untrained base model.
- **Does:** the full evaluation again at each step: the decision trials, the rule recovered from them, and the report
  questions. Nothing is trained.
- **Gives:** the same per-checkpoint files a training run writes, in `results/<run>/evaluate_only/<NAME>/`, and a check
  that the decisions match the run's own. The run's own files are not touched.
- **Use it to:** confirm that a run's results reproduce, or evaluate saved steps again under other evaluation settings.
- **Runs on its own:** the set-up cells clone and install the repo on a fresh machine, and the output is pushed to
  the Hub (`HUB_REPO`) when it is written.

## Details

Run the trials again on checkpoints a run already saved: **no training**. The base model loads once; each step
applies that step's saved LoRA adapter and goes through the same evaluation a training run gives its checkpoints:

1. the decision trials (`model_estimating.estimation_trials_per_persona`, 1000 for the one-persona runs), answered by
   the model,
2. the rule recovered from those answers,
3. the report questions, scored against the recovered rule and the true one.

Step `0` is the untrained base model. Everything comes from the run's own `results/<run>/settings.json`, so the
answers should match the run's: the **Reproduction** cell checks that, decision for decision.

| notebook | asks the decision trials? | asks report questions? | trains? |
|---|---|---|---|
| `run_experiment.ipynb` | yes | yes | yes |
| **`evaluate_checkpoints.ipynb`** (this one) | **yes** | **yes** | no |
| `ask_only.ipynb` | no (refits the rule from the run's saved decisions) | yes | no |

Needs, on this machine: `checkpoints/<run>/decision/step-<N>/` for each step and `results/<run>/settings.json`.
The **Download** cell fetches both from the Hub (`HUB_REPO`) when they are not here, with the run's saved decisions
for the reproduction check. `download_checkpoint.ipynb` does the same from a pasted link.

Writes only into `results/<run>/evaluate_only/<NAME>/`: the files a training run writes per checkpoint
(`selections_`, `recovered_`, `weight_reports_`, `reports_raw_`, `persona_scores_`, `learned_vs_reported_`, ...), a
`results.csv` of these steps alone, `settings.json` (what was asked) and `timing.csv`. Never appends to
`results/results.csv` and never changes a file of the run. The output folder is pushed to the Hub (`HUB_REPO`) at the
same path when the evaluation finishes, and again by the last cell.

Edit the settings cell only. An 8B run needs a GPU: about the evaluation time of one checkpoint of the run per step.

## Settings

Edit this cell only, then run the cells below it.

In [ ]:
RUN = "20261116-221120"     # a run id, or its timestamp prefix (results/<run>/ must hold settings.json)
STEPS = [0, 1500, 3000]     # saved decision steps; 0 = the base model, no adapter
NAME = "recheck"            # output folder: results/<run>/evaluate_only/<NAME>/
HUB_REPO = "adamchanna/bdp-plunkett-qwen3"   # the Hub repo: missing files are downloaded from it and the output is saved to it; None = neither
BRANCH = "bootstrap-ci"     # the branch of the repo to run: cloned on a fresh machine, brought up to date on an existing clone

# Everything below None / {} keeps the run's own settings, which is what the reproduction check expects.
QUESTIONS_FROM = None       # a settings file to take report_schema (the question set) from
ONLY = None                 # a list of question names, to ask only those
SHOW_NEITHER = None         # True / False: show option C in report trials
REPLY_PREFIX = None         # a string the reply starts with ("" = none)
OVERRIDES = {}              # {"block.field": value}, e.g. for a smaller card or a quicker pass:
                            #   {"compute.decision_eval_batch_size": 16, "compute.eval_batch_size": 16}
                            #   {"model_estimating.estimation_trials_per_persona": 200}
                            # On a CPU (slow; 0.6B smoke runs only):
                            #   {"compute.device": "cpu", "compute.mixed_precision": "no", "model_hyperparameters.quantization": "none"}

## Set up the machine

The same cells as `run_experiment.ipynb`, so this notebook runs on its own. On a fresh machine they clone the repo
(`BRANCH`) and install what it needs; on a machine that is already set up they bring the code up to date. The last
one asks for the Hugging Face token the save cells need.

In [ ]:
!nvidia-smi || echo "no nvidia-smi: CPU-only machine, an 8B run needs a GPU"

In [ ]:
# Clone BRANCH when the code is not already here, or move an existing clone onto it, then pin versions.
# run_me.py runs before the install: it re-pins requirements to whatever this machine
# already ships, keeping its CUDA-matched torch instead of letting pip swap it out.
import os, subprocess, sys

CLONE_URL = "https://github.com/Bilal-Trigui/Decision_Task_Database_Experiments.git"
if not os.path.exists("src"):
    if not os.path.exists("Decision_Task_Database_Experiments"):
        subprocess.run(["git", "clone", "-b", BRANCH, CLONE_URL], check=True)
    os.chdir("Decision_Task_Database_Experiments")
# An existing clone may sit on another branch or an older commit. This refuses rather than
# overwrite local edits: commit or stash them, or set BRANCH to the branch they are on.
subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
commit = subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print(f"code: branch {BRANCH}, commit {commit}")
subprocess.run([sys.executable, "run_me.py"], check=True)

# Keep model downloads in one folder, easy to delete on a shared disk.
os.environ.setdefault("HF_HOME", os.path.abspath("hf_cache"))
# Let the CUDA allocator grow segments instead of fragmenting. Must be set before
# torch initialises CUDA, which is why it lives here and not in the GPU check.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print("repo ready in", os.getcwd())

In [ ]:
# %pip installs into THIS kernel's environment. A pip subprocess can land packages
# somewhere the kernel cannot import from, which surfaces later as ModuleNotFoundError.
%pip install -q -r requirements.txt

In [ ]:
# Confirm the kernel can import what the pipeline needs.
#
# On JupyterHub, pip usually installs into the user site-packages, and a kernel
# that started before that folder existed does not have it on sys.path. That is
# the normal cause of ModuleNotFoundError straight after a successful install.
# Adding the folder here repairs it without a restart.
import importlib, importlib.util, os, site, sys

user_site = site.getusersitepackages()
if user_site and os.path.isdir(user_site) and user_site not in sys.path:
    sys.path.append(user_site)
    importlib.invalidate_caches()
    print("added user site-packages to sys.path:", user_site)

print("kernel python:", sys.executable)
needed = ("numpy", "pandas", "sklearn", "matplotlib", "torch", "transformers", "peft", "huggingface_hub")
missing = [m for m in needed if importlib.util.find_spec(m) is None]
if missing:
    print("MISSING:", missing)
    print("Restart the kernel (Kernel -> Restart Kernel), run the install cell, then this cell again.")
else:
    print("all imports available")

In [ ]:
# Confirm the GPU. The precision and the batch sizes are the run's own settings: bf16 needs Ampere or newer.
import torch

print("torch", torch.__version__)
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))
    print("vram GB", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
    print("bf16 supported", torch.cuda.is_bf16_supported())
else:
    print("no CUDA device: an 8B run needs a GPU")

In [ ]:
# Credentials. Typed rather than written into a cell, so nothing lands in git or in
# a saved notebook. Needed only to save to the Hub: downloading from a public repo and
# the public Qwen3 weights do not require a token.
#
# A token that the Hub rejects is discarded and you are asked again, so a typo or
# a revoked token never gets stuck in the environment.
import getpass, os
from huggingface_hub import HfApi
from huggingface_hub.utils import HfHubHTTPError

who = None
for attempt in range(3 if HUB_REPO else 0):
    token = os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face WRITE token (blank to skip): ")
    if not token:
        break
    try:
        who = HfApi(token=token).whoami()
        os.environ["HF_TOKEN"] = token
        break
    except HfHubHTTPError as err:
        os.environ.pop("HF_TOKEN", None)
        if err.response is not None and err.response.status_code == 401:
            print("The Hub rejected that token (401). Check it is a WRITE token and not revoked. Try again.")
            continue
        raise

if not HUB_REPO:
    print("HUB_REPO is None: nothing is downloaded from the Hub and nothing is saved to it")
elif who is None:
    print("no token: the notebook works, but saving refuses and nothing survives the machine")
else:
    owner = HUB_REPO.split("/")[0]
    allowed = [who["name"]] + [o["name"] for o in who.get("orgs", [])]
    print("token belongs to:", who["name"])
    if owner in allowed:
        print(f"HUB_REPO owner '{owner}' matches. Saving will work.")
    else:
        print(f"MISMATCH: HUB_REPO starts with '{owner}' but this token can only write to {allowed}.")
        print("Saving will be refused. Use a token of that account, or name a Hub repo you can write to in HUB_REPO.")

## Imports

In [ ]:
import os, sys, time
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "evaluate_saved.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("run this notebook from inside the project folder")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 120)

from src import ask_only as A
from src import evaluate_saved as ES
from src.persist import push_folder
print("project:", ROOT)

## Download (from the Hub)

Fetches what this notebook reads and this machine does not have yet, for `RUN` at `STEPS`: each step's adapter folder
(`checkpoints/<run>/decision/step-<N>/`), the run's `results/<run>/settings.json`, and its saved `selections_` file at
each step (what the Reproduction cell compares with). Step `0` needs no adapter. Files already here are kept, and
every file is checked against the Hub.

This is `download_checkpoint.ipynb`'s code, in `src/download.py`. To fetch anything else, pass its Hugging Face link:
`DL.fetch_links(["https://huggingface.co/<user>/<repo>/tree/main/checkpoints/<run>/decision/step-1500"])`.

In [ ]:
from src import download as DL

if HUB_REPO:
    RUN = DL.fetch_run(HUB_REPO, RUN, STEPS, results=("selections",))    # RUN becomes the full run id
else:
    print("HUB_REPO is None: nothing downloaded")

## Which adapters are saved

In [ ]:
run_dir = A.find_run(RUN)
saved = A.available_steps(run_dir.name)
print(run_dir.name)
print("saved steps here:", saved)
missing = [s for s in STEPS if s and s not in saved]
if missing:
    raise FileNotFoundError(f"steps {missing} have no adapter here: run the Download cell above (HUB_REPO must name the "
                            "Hub repo), or fetch them with download_checkpoint.ipynb")

## Evaluate

In [ ]:
LOADED = globals().get("LOADED")    # the base model, kept across runs of this cell (same model only)
t = time.time()
out_dir, timing, LOADED = ES.evaluate(RUN, STEPS, NAME, QUESTIONS_FROM, ONLY, SHOW_NEITHER, REPLY_PREFIX, OVERRIDES,
                                      loaded=LOADED)
print(out_dir, f"{time.time() - t:.0f}s")
# Saved to the Hub at the same path. If this stops (no token, no network) the output is still on disk:
# run the token cell and then the last cell of the notebook.
push_folder(out_dir, HUB_REPO)
pd.DataFrame(timing)

## Reproduction

In [ ]:
# The share of decisions equal to the run's own at each step. With the run's settings on the same kind of card this
# should be 1.0; another card, batch size or quantization can move a probability in the last digits and flip a
# near-tie, so a rate just under 1 with a small max_probability_gap is the hardware, not a different model.
pd.DataFrame([r for r in (ES.reproduction(run_dir, out_dir, s) for s in STEPS) if r])

## Results

In [ ]:
rows = pd.read_csv(out_dir / "results.csv")
cols = [c for c in ["stage", "checkpoint_step", "block", "question", "decision_accuracy", "parse_rate",
                    "faithfulness", "faithfulness_shuffled", "reported_vs_hidden"] if c in rows.columns]
rows[cols]

In [ ]:
# The persona readouts and the trial regression on what was just asked, as for a training run's folder.
import subprocess
for tool in (["tools/persona_readouts.py", str(out_dir), "--boot", "2000"], ["tools/trial_regression.py", str(out_dir)]):
    subprocess.run([sys.executable, *tool], check=False)

## Save it off the machine

The Evaluate cell has already pushed its output folder. This cell pushes every `results/<run>/evaluate_only/` folder
on this machine again, with what the cells after it added (the readouts and the trial regression), so nothing is left
behind: run it before you leave a rented machine. It stops with the reason when the Hub refuses.

In [ ]:
folders = sorted((ROOT / "results").glob("*/evaluate_only"))
for folder in folders:
    push_folder(folder, HUB_REPO)
print(f"{len(folders)} evaluate_only folder(s) on this machine" + ("" if HUB_REPO else "; HUB_REPO is None, so none was saved"))